# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print("{}: {}".format(metadata.name, metadata.description))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @ids
print("Available record sets:")
for record_set in metadata.record_set:
    print(f"- Name: {record_set.name}, @id: {record_set.id}")
    if hasattr(record_set, 'field'):
        print("  Fields:")
        for field in record_set.field:
            print(f"    - Field name: {field.name}, @id: {field.id}, Data type: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets = [rs.id for rs in metadata.record_set]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded records for record set {record_set_id} (shape: {dataframes[record_set_id].shape})")

# For demonstration, pick the first record set as primary
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"\nColumns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose numeric and grouping fields by inspecting the DataFrame columns
# For demonstration, we attempt common proteomics columns such as 'MW' (molecular weight) or 'abundance', adjust if needed.
numeric_candidates = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col])]
print('Numeric field candidates:', numeric_candidates)
numeric_field = None
for col in ['MW', 'Abundance', 'abundance', 'coverage_percentage']:
    if col in dataframes[main_record_set_id].columns:
        numeric_field = col
        break
if numeric_field is None and numeric_candidates:
    numeric_field = numeric_candidates[0]
# Fallback
if numeric_field:
    print(f"Using numeric field '{numeric_field}' for analysis.")
    # Remove non-numeric, handle missing if needed
    df_numeric = pd.to_numeric(dataframes[main_record_set_id][numeric_field], errors='coerce')

    threshold = df_numeric.mean() if pd.notnull(df_numeric.mean()) else 0
    filtered_df = dataframes[main_record_set_id][df_numeric > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (df_numeric[filtered_df.index] - df_numeric.mean()) / df_numeric.std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt grouping by a categorical field such as 'description' or similar
    group_fields = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype == object and col.lower() != 'name']
    group_field = None
    for col in ['description', 'group', 'sample', 'accession']:
        if col in dataframes[main_record_set_id].columns:
            group_field = col
            break
    if not group_field and group_fields:
        group_field = group_fields[0]
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouping field exists, boxplot group vs numeric
    if group_field:
        plt.figure(figsize=(10,4))
        # Limit number of groups for clarity
        top_groups = filtered_df[group_field].value_counts().head(10).index
        sns.boxplot(data=filtered_df[filtered_df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} distribution by {group_field} (top 10)')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded the FAIRˆ2 (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) dataset using `mlcroissant`.
* We listed available record sets and fields using `@id` references for transparency and reproducibility.
* Numeric and categorical candidate fields were selected based on their data types and names. Filtering, normalization, and group-based aggregation were demonstrated.
* Distributions and group comparisons were visualized to gain a first intuition of the data.
* Refer to the dataset's Croissant schema and publication for detailed semantics of each field.